In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import unicodedata
import xagg


In [2]:
nuts = gpd.read_file(r'/Volumes/Dhruv_External_Disk/portugal_borders/NUTS_RG_20M_2024_4326.shp/NUTS_RG_20M_2024_4326.shp')
# Function to normalize the names
def normalize_municipality_name(name):
    if type(name) == float:
        return name
    else:
        # Normalize the string (remove diacritical marks)
        name_without_diacritics = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
        # Convert to lowercase
        return name_without_diacritics.lower()

nuts['NUTS_NAME'] = nuts['NUTS_NAME'].map(normalize_municipality_name)

nuts_iberia_2 = nuts[(nuts['CNTR_CODE'].isin(['PT', 'ES'])) &  (nuts['LEVL_CODE'] == 2) & (~nuts['NUTS_NAME'].isin(['canarias', 'regiao autonoma dos acores', 'regiao autonoma da madeira', 'illes balears']))]
west, south, east, north = nuts_iberia_2.total_bounds


In [3]:
nuts_iberia_3 = nuts[nuts['NUTS_NAME'].isin(nuts_iberia_2['NUTS_NAME'])][['NUTS_ID', 'NUTS_NAME', 'geometry']].copy()

In [4]:
IBERIA_BBOX = {
    "min_lat": south,  # Minimum latitude
    "max_lat": north,  # Maximum latitude
    "min_lon": west,   # Minimum longitude
    "max_lon": east    # Maximum longitude
}
IBERIA_BBOX

{'min_lat': 35.271704476000025,
 'max_lat': 43.735114829000054,
 'min_lon': -9.490672837999966,
 'max_lon': 3.25861242000002}

## Temperature Daily Indicators

In [8]:
from pathlib import Path
from tqdm.auto import tqdm

# Load all temperature daily indicator files for 2003-2023
t2m_folder = Path("/Volumes/Dhruv_External_Disk/climate_data/temperature/daily_data/indicators")
years = range(2003, 2024)  # 2003 to 2023 inclusive

# Load and filter each year's data
t2m_ds_list = []
for year in tqdm(years, desc="Loading T2M data"):
    nc_file = t2m_folder / f"t2m_daily_indicators_{year}.nc"
    ds = xr.open_dataset(nc_file)
    
    # Filter to Iberia bounding box
    ds_filtered = ds.sel(
        latitude=slice(IBERIA_BBOX["max_lat"], IBERIA_BBOX["min_lat"]),  # Note: descending order for latitude
        longitude=slice(IBERIA_BBOX["min_lon"], IBERIA_BBOX["max_lon"])
    )
    
    t2m_ds_list.append(ds_filtered)

# Concatenate all years into one dataset
t2m_ds = xr.concat(t2m_ds_list, dim="datetime")
print(f"✓ Loaded T2M data: {t2m_ds.dims}")

# Aggregate to NUTS 3 regions using xagg
print("Aggregating T2M data to NUTS 3 regions...")
t2m_agg = [
    xagg.aggregate(ds, xagg.pixel_overlaps(ds, nuts_iberia_3)).to_dataframe()
    for ds in tqdm(t2m_ds_list, desc="Aggregating T2M")
]

# Concatenate all aggregated DataFrames
t2m_df = pd.concat(t2m_agg, ignore_index=False)
print(f"✓ T2M aggregation complete: {t2m_df.shape}")
t2m_df.head()

Loading T2M data:   0%|          | 0/21 [00:00<?, ?it/s]

✓ Loaded T2M data: FrozenMappingWarningOnValuesAccess({'latitude': 84, 'longitude': 127, 'datetime': 7712})
Aggregating T2M data to NUTS 3 regions...


Aggregating T2M:   0%|          | 0/21 [00:00<?, ?it/s]

creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
success!
aggregating tmax...
aggregating tmax...
aggregating tmin...
aggregating tmin...
aggregating tavg...
aggregating tavg...
aggregating dtr...
aggregating dtr...
aggregating tstd...
aggregating tstd...
aggregating tropical_night...
aggregating tropical_night...
aggregating frost_day...
aggregating frost_day...
aggregating deg_hours_above_30...
aggregating deg_hours_above_30...
aggregating deg_hours_above_35...
aggregating deg_hours_above_35...
aggregating deg_hours_above_40...
aggregating deg_hours_above_40...
aggregating deg_hours_below_0...
aggregating deg_hours_below_0...
aggregating hot_day_30...
aggregating hot_day_30...
aggregating hot_day_35...
aggregating hot_day_35...
aggregating hot_day_40...
aggregating hot_day_40...
aggregating extremely_hot_p90...
aggregating extremely_hot_p90...
aggregating extremely_hot_p9

NUTS_ID  NUTS_NAME       tmax      tmin       tavg  \
poly_idx datetime                                                        
0        2002-12-31   ES130  cantabria   0.000000  0.000000   0.000000   
         2003-01-01   ES130  cantabria  13.519130  7.221061  10.779487   
         2003-01-02   ES130  cantabria  14.085873  7.793892  11.858628   
         2003-01-03   ES130  cantabria  11.521258  4.866863   8.420573   
         2003-01-04   ES130  cantabria   6.831613  2.980502   4.923213   

                          dtr      tstd  tropical_night  frost_day  \
poly_idx datetime                                                    
0        2002-12-31  0.000000  0.000000             0.0        0.0   
         2003-01-01  6.298069  2.361062             0.0        0.0   
         2003-01-02  6.291981  1.859720             0.0        0.0   
         2003-01-03  6.654395  1.919474             0.0        0.0   
         2003-01-04  3.851111  1.161391             0.0        0.0   

                     deg_hours_above_30  deg_hours_above_35  \
poly_idx datetime                                             
0        2002-12-31                 0.0                 0.0   
         2003-01-01                 0.0                 0.0   
         2003-01-02                 0.0                 0.0   
         2003-01-03                 0.0                 0.0   
         2003-01-04                 0.0                 0.0   

                     deg_hours_above_40  deg_hours_below_0  hot_day_30  \
poly_idx datetime                                                        
0        2002-12-31                 0.0           0.000000         0.0   
         2003-01-01                 0.0           0.000000         0.0   
         2003-01-02                 0.0           0.000000         0.0   
         2003-01-03                 0.0           0.033375         0.0   
         2003-01-04                 0.0           0.268710         0.0   

                     hot_day_35  hot_day_40  extremely_hot_p90  \
poly_idx datetime                                                
0        2002-12-31         0.0         0.0                0.0   
         2003-01-01         0.0         0.0                0.0   
         2003-01-02         0.0         0.0                0.0   
         2003-01-03         0.0         0.0                0.0   
         2003-01-04         0.0         0.0                0.0   

                     extremely_hot_p95  extremely_hot_p99  temp_anomaly  
poly_idx datetime                                                        
0        2002-12-31                0.0                0.0      0.000000  
         2003-01-01                0.0                0.0      0.674561  
         2003-01-02                0.0                0.0      1.753701  
         2003-01-03                0.0                0.0     -1.684354  
         2003-01-04                0.0                0.0     -5.181713

## Precipitation Daily Indicators

In [9]:
# Load all precipitation daily indicator files for 2003-2023
tp_folder = Path("/Volumes/Dhruv_External_Disk/climate_data/precipitation/historical/daily/indicators")
years = range(2003, 2024)  # 2003 to 2023 inclusive

# Load and filter each year's data
tp_ds_list = []
for year in tqdm(years, desc="Loading TP data"):
    nc_file = tp_folder / f"tp_daily_indicators_{year}.nc"
    ds = xr.open_dataset(nc_file)
    
    # Filter to Iberia bounding box
    ds_filtered = ds.sel(
        latitude=slice(IBERIA_BBOX["max_lat"], IBERIA_BBOX["min_lat"]),  # Note: descending order for latitude
        longitude=slice(IBERIA_BBOX["min_lon"], IBERIA_BBOX["max_lon"])
    )
    
    tp_ds_list.append(ds_filtered)

# Concatenate all years into one dataset
tp_ds = xr.concat(tp_ds_list, dim="datetime")
print(f"✓ Loaded TP data: {tp_ds.dims}")

# Aggregate to NUTS 3 regions using xagg
print("Aggregating TP data to NUTS 3 regions...")
tp_agg = [
    xagg.aggregate(ds, xagg.pixel_overlaps(ds, nuts_iberia_3)).to_dataframe()
    for ds in tqdm(tp_ds_list, desc="Aggregating TP")
]

# Concatenate all aggregated DataFrames
tp_df = pd.concat(tp_agg, ignore_index=False)
print(f"✓ TP aggregation complete: {tp_df.shape}")
tp_df.head()

Loading TP data:   0%|          | 0/21 [00:00<?, ?it/s]

✓ Loaded TP data: FrozenMappingWarningOnValuesAccess({'latitude': 84, 'longitude': 127, 'datetime': 7712})
Aggregating TP data to NUTS 3 regions...


Aggregating TP:   0%|          | 0/21 [00:00<?, ?it/s]

creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
success!
aggregating tp_total...
aggregating tp_total...
aggregating tp_max...
aggregating tp_max...
aggregating tp_std...
aggregating tp_std...
aggregating wet_day...
aggregating wet_day...
aggregating dry_day...
aggregating dry_day...
aggregating rx10mm...
aggregating rx10mm...
aggregating rx20mm...
aggregating rx20mm...
aggregating extremely_wet_p90...
aggregating extremely_wet_p90...
aggregating extremely_wet_p95...
aggregating extremely_wet_p95...
aggregating extremely_wet_p99...
aggregating extremely_wet_p99...
aggregating extreme_prec_p90...
aggregating extreme_prec_p90...
aggregating extreme_prec_p95...
aggregating extreme_prec_p95...
aggregating extreme_prec_p99...
aggregating extreme_prec_p99...
aggregating prec_anomaly...
aggregating prec_anomaly...
all variables aggregated to polygons!
all variables aggregated to 

NUTS_ID  NUTS_NAME   tp_total    tp_max    tp_std  \
poly_idx datetime                                                       
0        2002-12-31   ES130  cantabria   0.000000  0.000000  0.000000   
         2003-01-01   ES130  cantabria  29.397751  1.918484  0.696189   
         2003-01-02   ES130  cantabria  10.284085  2.211214  0.527076   
         2003-01-03   ES130  cantabria   7.765035  2.164129  0.421391   
         2003-01-04   ES130  cantabria  33.948182  3.689381  1.499405   

                      wet_day   dry_day    rx10mm    rx20mm  \
poly_idx datetime                                             
0        2002-12-31  0.000000  1.000000  0.000000  0.000000   
         2003-01-01  0.904607  0.095393  0.904607  0.700268   
         2003-01-02  0.904607  0.095393  0.393721  0.133013   
         2003-01-03  0.904607  0.095393  0.269722  0.000000   
         2003-01-04  0.904607  0.095393  0.904607  0.826505   

                     extremely_wet_p90  extremely_wet_p95  extremely_wet_p99  \
poly_idx datetime                                                              
0        2002-12-31           0.000000                0.0                0.0   
         2003-01-01           0.000134                0.0                0.0   
         2003-01-02           0.000000                0.0                0.0   
         2003-01-03           0.000000                0.0                0.0   
         2003-01-04           0.000000                0.0                0.0   

                     extreme_prec_p90  extreme_prec_p95  extreme_prec_p99  \
poly_idx datetime                                                           
0        2002-12-31          0.000000               0.0               0.0   
         2003-01-01          0.000812               0.0               0.0   
         2003-01-02          0.000000               0.0               0.0   
         2003-01-03          0.000000               0.0               0.0   
         2003-01-04          0.000000               0.0               0.0   

                     prec_anomaly  
poly_idx datetime                  
0        2002-12-31    -34.844856  
         2003-01-01     -5.447105  
         2003-01-02    -24.560771  
         2003-01-03    -27.079821  
         2003-01-04     -0.896674

## Air Pollutants Daily Data

In [10]:
# Load all air pollutant daily files for 2003-2023
pollutants = {
    'black_carbon': '/Volumes/Dhruv_External_Disk/climate_data/air_pollutants/black_carbon/daily',
    'methane': '/Volumes/Dhruv_External_Disk/climate_data/air_pollutants/methane/daily',
    'nitrogen_dioxide': '/Volumes/Dhruv_External_Disk/climate_data/air_pollutants/nitrogen_dioxide/daily',
    'nitrogen_monoxide': '/Volumes/Dhruv_External_Disk/climate_data/air_pollutants/nitrogen_monoxide/daily',
    'ozone': '/Volumes/Dhruv_External_Disk/climate_data/air_pollutants/ozone/daily'
}

years = range(2003, 2024)  # 2003 to 2023 inclusive

# Dictionary to store datasets for each pollutant
pollutant_ds_dict = {}

for pollutant_name, folder_path in pollutants.items():
    print(f"\n{'='*60}")
    print(f"Processing {pollutant_name.upper()}")
    print(f"{'='*60}")
    
    folder = Path(folder_path)
    ds_list = []
    
    for year in tqdm(years, desc=f"Loading {pollutant_name}"):
        nc_file = folder / f"{pollutant_name}_daily_{year}.nc"
        
        if nc_file.exists():
            ds = xr.open_dataset(nc_file)
            
            # Filter to Iberia bounding box
            # Check coordinate names (might be 'time' instead of 'datetime')
            if 'latitude' in ds.coords and 'longitude' in ds.coords:
                ds_filtered = ds.sel(
                    latitude=slice(IBERIA_BBOX["max_lat"], IBERIA_BBOX["min_lat"]),
                    longitude=slice(IBERIA_BBOX["min_lon"], IBERIA_BBOX["max_lon"])
                )
            else:
                # Handle alternate coordinate names if needed
                ds_filtered = ds
            
            ds_list.append(ds_filtered)
        else:
            print(f"Warning: File not found - {nc_file}")
    
    if ds_list:
        # Concatenate all years
        time_dim = 'time' if 'time' in ds_list[0].dims else 'datetime'
        pollutant_ds = xr.concat(ds_list, dim=time_dim)
        pollutant_ds_dict[pollutant_name] = {
            'dataset': pollutant_ds,
            'ds_list': ds_list
        }
        print(f"✓ Loaded {pollutant_name} data: {pollutant_ds.dims}")

# Aggregate each pollutant to NUTS 3 regions
pollutant_dfs = {}

for pollutant_name, data in pollutant_ds_dict.items():
    print(f"\nAggregating {pollutant_name} to NUTS 3 regions...")
    ds_list = data['ds_list']
    
    agg_list = [
        xagg.aggregate(ds, xagg.pixel_overlaps(ds, nuts_iberia_3)).to_dataframe()
        for ds in tqdm(ds_list, desc=f"Aggregating {pollutant_name}")
    ]
    
    # Concatenate all aggregated DataFrames
    df = pd.concat(agg_list, ignore_index=False)
    pollutant_dfs[pollutant_name] = df
    print(f"✓ {pollutant_name} aggregation complete: {df.shape}")

# Display summary
print(f"\n{'='*60}")
print("SUMMARY OF AIR POLLUTANT DATA")
print(f"{'='*60}")
for pollutant_name, df in pollutant_dfs.items():
    print(f"{pollutant_name}: {df.shape}")
    
# Show sample of first pollutant
first_pollutant = list(pollutant_dfs.keys())[0]
print(f"\nSample from {first_pollutant}:")
pollutant_dfs[first_pollutant].head()


Processing BLACK_CARBON


Loading black_carbon:   0%|          | 0/21 [00:00<?, ?it/s]

✓ Loaded black_carbon data: FrozenMappingWarningOnValuesAccess({'time': 7670, 'latitude': 11, 'longitude': 16})

Processing METHANE


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_8328/1822944731.py:27: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(nc_file)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_8328/1822944731.py:27: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(nc_file)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_8328/1822944731.py:27: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(nc_file)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_8328/

Loading methane:   0%|          | 0/21 [00:00<?, ?it/s]

...

Aggregating nitrogen_monoxide:   0%|          | 0/21 [00:00<?, ?it/s]

creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
aggregating no_daily_mean...
aggregating no_daily_max...
all variables aggregated to polygons!
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
calculating overlaps between pixels and output polygons...
success!
aggregating no_daily_mean...
aggregating no_daily_max...
all variables aggregated to polygons!
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
aggregating no_daily_mean...
aggregating no_daily_max...
all variables aggregated to polygons!
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
aggr

Aggregating ozone:   0%|          | 0/21 [00:00<?, ?it/s]

creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
aggregating o3_daily_mean...
aggregating o3_daily_max...
all variables aggregated to polygons!
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
aggregating o3_daily_mean...
aggregating o3_daily_max...
all variables aggregated to polygons!
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
all variables aggregated to polygons!
creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
aggregating o3_daily_mean...
aggregating o3_daily_max...
all variables aggregated to polygons!
creating polygons for each pixel..

NUTS_ID  NUTS_NAME  bcaod550_daily_mean
poly_idx time                                              
0        2003-01-01   ES130  cantabria             0.002657
         2003-01-02   ES130  cantabria             0.002628
         2003-01-03   ES130  cantabria             0.001175
         2003-01-04   ES130  cantabria             0.000827
         2003-01-05   ES130  cantabria             0.001448

## Merge All Data

In [13]:
# Prepare T2M dataframe
t2m_clean = t2m_df.reset_index()
if 'poly_idx' in t2m_clean.columns:
    t2m_clean = t2m_clean.drop(columns=['poly_idx'])

# Prepare TP dataframe
tp_clean = tp_df.reset_index()
if 'poly_idx' in tp_clean.columns:
    tp_clean = tp_clean.drop(columns=['poly_idx'])

# Prepare pollutant dataframes
pollutant_clean = {}
for pollutant_name, df in pollutant_dfs.items():
    df_clean = df.reset_index()
    if 'poly_idx' in df_clean.columns:
        df_clean = df_clean.drop(columns=['poly_idx'])
    pollutant_clean[pollutant_name] = df_clean

# Identify merge keys (check column names)
# Assuming columns are 'datetime', 'NUTS_ID', 'NUTS_NAME' or similar variants
print("T2M columns:", t2m_clean.columns.tolist()[:10])
print("TP columns:", tp_clean.columns.tolist()[:10])
print("First pollutant columns:", pollutant_clean[list(pollutant_clean.keys())[0]].columns.tolist()[:10])

# Standardize column names if needed (looking for datetime/time and NUTS identifiers)
# Merge T2M and TP first
merged_df = pd.merge(
    t2m_clean, 
    tp_clean, 
    on=['datetime', 'NUTS_ID', 'NUTS_NAME'],
    how='outer',
    suffixes=('_t2m', '_tp')
)

print(f"After merging T2M + TP: {merged_df.shape}")

# Merge each pollutant
for pollutant_name, df_clean in pollutant_clean.items():
    # Check if datetime column exists, otherwise might be 'time'
    if 'datetime' not in df_clean.columns and 'time' in df_clean.columns:
        df_clean = df_clean.rename(columns={'time': 'datetime'})
    
    merged_df = pd.merge(
        merged_df,
        df_clean,
        on=['datetime', 'NUTS_ID', 'NUTS_NAME'],
        how='outer',
        suffixes=('', f'_{pollutant_name}')
    )
    print(f"After merging {pollutant_name}: {merged_df.shape}")


merged_df.head()

T2M columns: ['datetime', 'NUTS_ID', 'NUTS_NAME', 'tmax', 'tmin', 'tavg', 'dtr', 'tstd', 'tropical_night', 'frost_day']
TP columns: ['datetime', 'NUTS_ID', 'NUTS_NAME', 'tp_total', 'tp_max', 'tp_std', 'wet_day', 'dry_day', 'rx10mm', 'rx20mm']
First pollutant columns: ['time', 'NUTS_ID', 'NUTS_NAME', 'bcaod550_daily_mean']
After merging T2M + TP: (233760, 35)
After merging black_carbon: (233760, 36)
After merging methane: (233760, 38)
After merging nitrogen_dioxide: (233760, 40)
After merging nitrogen_monoxide: (233760, 42)
After merging ozone: (233760, 44)
After merging nitrogen_monoxide: (233760, 42)
After merging ozone: (233760, 44)


,datetime,NUTS_ID,NUTS_NAME,tmax,tmin,tavg,dtr,tstd,tropical_night,frost_day,...,prec_anomaly,bcaod550_daily_mean,ch4_daily_mean,ch4_daily_max,no2_daily_mean,no2_daily_max,no_daily_mean,no_daily_max,o3_daily_mean,o3_daily_max
0,2002-12-31,ES130,cantabria,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,-34.844856,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2003-01-01,ES130,cantabria,13.519130,7.221061,10.779487,6.298069,2.361062,0.0,0.0,...,-5.447105,0.002657,0.009410,0.009452,0.000003,0.000004,2.224838e-07,8.118575e-07,0.006566,0.006718
2,2003-01-02,ES130,cantabria,14.085873,7.793892,11.858628,6.291981,1.859720,0.0,0.0,...,-24.560771,0.002628,0.009385,0.009415,0.000003,0.000003,1.984091e-07,6.926368e-07,0.006492,0.006753
3,2003-01-03,ES130,cantabria,11.521258,4.866863,8.420573,6.654395,1.919474,0.0,0.0,...,-27.079821,0.001175,0.009434,0.009445,0.000003,0.000004,2.289410e-07,7.955644e-07,0.007435,0.007846
4,2003-01-04,ES130,cantabria,6.831613,2.980502,4.923213,3.851111,1.161391,0.0,0.0,...,-0.896674,0.000827,0.009481,0.009524,0.000003,0.000004,2.825359e-07,9.077644e-07,0.007330,0.007847


In [17]:
merged_df = merged_df[merged_df['datetime'] != '2002-12-31 00:00:00']

In [ ]:
merged_df.to_parquet(r'iberia_daily_weather.parquet')